In [ ]:
import pandas as pd

# from scouting.data_transform.xthreat import compute_xthreat

spadl_data = pd.read_parquet("../tests/spadl_salzbourg_sb29_game_data.parquet")
player_game_stats = pd.read_parquet("../tests/player_game_stats_salzbourg_sb29.parquet")
spadl_data

In [32]:
game = 1866220

### Compute features for VAEP

In [33]:
import socceraction.vaep.features as fs
import socceraction.vaep.labels as lab
import socceraction.vaep.formula as vaepformula


In [34]:
xfns = [
    fs.actiontype,
    fs.actiontype_onehot,
    fs.bodypart,
    fs.bodypart_onehot,
    fs.result,
    fs.result_onehot,
    fs.goalscore,
    fs.startlocation,
    fs.endlocation,
    fs.movement,
    fs.space_delta,
    fs.startpolar,
    fs.endpolar,
    fs.team,
    fs.time,
    fs.time_delta
]


In [35]:
nb_prev_actions = 3 # Default
team_id = 2332 # Brest

In [ ]:
gamestates = fs.gamestates(spadl_data, nb_prev_actions)
gamestates = fs.play_left_to_right(gamestates, team_id)

X = pd.concat([fn(gamestates) for fn in xfns], axis=1)
X

In [ ]:
yfns = [lab.scores, lab.concedes, lab.goal_from_shot]

Y = pd.concat([fn(spadl_data) for fn in yfns], axis=1)
Y

In [38]:
# from scouting.constants import ROOT_PATH

# X.to_parquet(ROOT_PATH / "tests" / "VAEP_features.parquet")
# Y.to_parquet(ROOT_PATH / "tests" / "VAEP_labels.parquet")


### Estimate scoring and conceding probabilities

⚠️ **There should be a train and a test set.**

In [ ]:
# Feature selection
xfns = [
    fs.actiontype,
    fs.actiontype_onehot,
    #fs.bodypart,
    fs.bodypart_onehot,
    fs.result,
    fs.result_onehot,
    fs.goalscore,
    fs.startlocation,
    fs.endlocation,
    fs.movement,
    fs.space_delta,
    fs.startpolar,
    fs.endpolar,
    fs.team,
    #fs.time,
    fs.time_delta,
    #fs.actiontype_result_onehot
]

Xcols = fs.feature_column_names(xfns, nb_prev_actions)

X_train = X[Xcols]
Y_train = Y
X_train

In [ ]:
Y_train

In [41]:
# 3. train classifier for VAEP
import xgboost

preds = pd.DataFrame()
models = {}
for col in list(Y_train.columns):
    model = xgboost.XGBClassifier(n_estimators=50, max_depth=3, n_jobs=-3, verbosity=1, enable_categorical=True)
    model.fit(X_train, Y_train[col])
    models[col] = model

In [ ]:
from sklearn.metrics import brier_score_loss, roc_auc_score, log_loss

# Should have a proper test set but as I only run things on a single game, it's ok for now
testX, testY = X_train, Y_train

def evaluate(y, y_hat):
    p = sum(y) / len(y)
    base = [p] * len(y)
    brier = brier_score_loss(y, y_hat)
    print(f"  Brier score: %.5f (%.5f)" % (brier, brier / brier_score_loss(y, base)))
    ll = log_loss(y, y_hat)
    print(f"  log loss score: %.5f (%.5f)" % (ll, ll / log_loss(y, base)))
    print(f"  ROC AUC: %.5f" % roc_auc_score(y, y_hat))

for col in testY.columns:
    preds[col] = [p[1] for p in models[col].predict_proba(testX)]
    print(f"### Y: {col} ###")
    evaluate(testY[col], preds[col])

In [ ]:
preds

In [44]:
# from scouting.constants import ROOT_PATH

# spadl_data_with_vaep_preds = pd.concat([spadl_data, preds], axis=1)
# spadl_data_with_vaep_preds.to_parquet(ROOT_PATH / "tests" / "vaep_preds_salzbourg_brest.parquet")

### Best VAEP actions

In [ ]:
preds

In [ ]:
A = []


values = vaepformula.value(spadl_data, preds.scores, preds.concedes)
A.append(pd.concat([spadl_data, preds, values], axis=1))


# Concert vaep values for many games in a for loop ?
A = pd.concat(A).sort_values(["game_id", "period_id", "time_seconds"]).reset_index(drop=True)
A

In [ ]:
# Most valuable players

A["count"] = 1

# Compute each player's number of actions and total VAEP values
playersR = (
    A[["player_id", "player", "vaep_value", "offensive_value", "defensive_value", "count"]]
    .groupby(["player_id", "player"])
    .sum()
    .reset_index()
)

# Show results
playersR = playersR[["player_id", "player", "vaep_value", "offensive_value", "defensive_value", "count"]]
playersR.sort_values("vaep_value", ascending=False)[:20]

In [ ]:
pd.DataFrame(raw_events[1866220])

In [ ]:
# Normalize for minutes played
# pg = pg[pg.game_id.isin(games.game_id)]
mp = player_game_stats[["player", "minutes_played"]].groupby("player").sum().reset_index()

stats = playersR.merge(mp, on="player")
# stats = stats[stats.minutes_played > 180] # at least two full games played
stats["vaep_rating"] = stats.vaep_value * 90 / stats.minutes_played
stats["offensive_rating"] = stats.offensive_value * 90 / stats.minutes_played
stats["defensive_rating"] = stats.defensive_value * 90 / stats.minutes_played
stats.sort_values("vaep_rating",ascending=False)[:10]

### Get Most stylish actions

In [ ]:
sorted_A

In [ ]:
A.columns

In [ ]:
import matplotsoccer

sorted_A = A.sort_values("vaep_value", ascending=False)
sorted_A = sorted_A[sorted_A.away_team == "Brest"] # view only actions from Brest
sorted_A = sorted_A[~sorted_A.type_name.str.contains("shot")] #eliminate shots

def get_time(period_id,time_seconds):
    m = int((period_id-1)*45 + time_seconds // 60)
    s = int(time_seconds % 60)
    return f"{m}m{s}s"

for j in range(0, 10):
    row = list(sorted_A[j:j+1].itertuples())[0]
    i = row.Index
    a = A[i - 3 : i+2].copy()
        
    # g = list(games[games.game_id == a.game_id.values[0]].itertuples())[0]
    # game_info = f"{g.game_date} {g.home_team} {g.home_score}-{g.away_score} {g.away_team}"
    # minute = int((row.period_id-1)*45 + row.time_seconds // 60)
    # print(f"{game_info} {minute}' {row.type_name} {row.player_name}")

    # a["scores"] = a.scores.apply(lambda x : "%.3f" % x )
    # a["vaep_value"] = a.vaep_value.apply(lambda x : "%.3f" % x )
    cols = ["minute", "type_name", "player", "away_team", "scores", "vaep_value"]
    matplotsoccer.actions(a[["start_x", "start_y", "end_x",  "end_y"]],
                a.type_name,
                team=a.away_team,
                result = a.result_name == "success",
                label=a[cols],
                labeltitle = cols,
                zoom=False)